In [27]:
import pandas as pd
import networkx as nx
import requests
import pubchempy as pcp
from xml.etree import ElementTree as ET
import time
import csv
from tqdm import tqdm

In [28]:
df_cg = pd.read_csv('../edges/chemgene.csv')
df_cd = pd.read_csv('../edges/chemdis.csv')
df_gd = pd.read_csv('../edges/genedis.csv')

df_c = pd.read_csv('../vocab/CTD_chemicals.csv')
df_d = pd.read_csv('../vocab/CTD_diseases.csv')
df_g = pd.read_csv('../vocab/CTD_genes.csv')

df_cg = df_cg[(df_cg['OrganismID'] == 9606)].drop_duplicates(subset=['ChemicalID', 'GeneID'])
df_cd = df_cd.drop_duplicates(subset=['ChemicalID', 'DiseaseName'])  
df_gd = df_gd.drop_duplicates(subset=['GeneID', 'DiseaseName'])

In [29]:
print(df_cg.info())
print(df_cd.info())
print(df_gd.info())

<class 'pandas.core.frame.DataFrame'>
Index: 812241 entries, 0 to 2993539
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ChemicalName        812241 non-null  object 
 1   ChemicalID          812241 non-null  object 
 2   CasRN               557625 non-null  object 
 3   GeneSymbol          812240 non-null  object 
 4   GeneID              812241 non-null  int64  
 5   GeneForms           809771 non-null  object 
 6   Organism            812241 non-null  object 
 7   OrganismID          812241 non-null  float64
 8   Interaction         812241 non-null  object 
 9   InteractionActions  812241 non-null  object 
 10  PubMedIDs           812241 non-null  object 
dtypes: float64(1), int64(1), object(9)
memory usage: 74.4+ MB
None
<class 'pandas.core.frame.DataFrame'>
Index: 3467034 entries, 0 to 9515672
Data columns (total 10 columns):
 #   Column               Dtype  
---  ------               -----  

In [30]:
known = pd.read_csv('chemsmiles.csv')
known.head()

,meshID,cids,smiles
0,C070055,35823,C1=CC(=C(C=C1C2=CC(=C(C=C2Cl)Cl)Cl)Cl)Cl
1,C024713,12389,CCCCCCCCCCCCCC
2,D000077612,166548,CCCCCOC1=CC=C(C=C1)C2=CC=C(C=C2)C3=CC=C(C=C3)C...
3,C538809,131634859,B(=O)OB1OB2OB(OB(O2)O1)[O-].[Na+]
4,C000626479,7565,C1=CC=C(C=C1)OC2=CC=C(C=C2)Br


In [31]:
df_cg = df_cg[df_cg["ChemicalID"].isin(known["meshID"])]
df_cd = df_cd[df_cd["ChemicalID"].isin(known["meshID"])]

print(df_cg.info())
print(df_cd.info())

<class 'pandas.core.frame.DataFrame'>
Index: 703832 entries, 0 to 2993539
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ChemicalName        703832 non-null  object 
 1   ChemicalID          703832 non-null  object 
 2   CasRN               551580 non-null  object 
 3   GeneSymbol          703831 non-null  object 
 4   GeneID              703832 non-null  int64  
 5   GeneForms           701695 non-null  object 
 6   Organism            703832 non-null  object 
 7   OrganismID          703832 non-null  float64
 8   Interaction         703832 non-null  object 
 9   InteractionActions  703832 non-null  object 
 10  PubMedIDs           703832 non-null  object 
dtypes: float64(1), int64(1), object(9)
memory usage: 64.4+ MB
None
<class 'pandas.core.frame.DataFrame'>
Index: 2911974 entries, 1 to 9515672
Data columns (total 10 columns):
 #   Column               Dtype  
---  ------               -----  

In [32]:
G = nx.Graph()


In [33]:
chemicals = (set(df_cg['ChemicalID'].unique()) | set(df_cd['ChemicalID'].unique())) & set(known['meshID'])
genes = set(df_cg['GeneID'].unique()) | set(df_gd['GeneID'].unique())
diseases = set(df_cd['DiseaseID'].unique()) | set(df_gd['DiseaseID'].unique())

print(len(chemicals))
print(len(genes))
diseases = [dis[5:] for dis in diseases]
print(len(diseases))
print(len(chemicals)+len(genes)+len(diseases))

13943
28151
7239
49333


In [34]:
for chem in chemicals:
    G.add_node(chem, node_type='chemical')
for gene in genes:
    G.add_node(gene, node_type='gene')
for disease in diseases:
    G.add_node(disease, node_type='disease')

for _, row in df_cg.iterrows():
    G.add_edge(row['ChemicalID'], row['GeneID'], edge_type='chem_gene')
for _, row in df_cd.iterrows():
    G.add_edge(row['ChemicalID'], row['DiseaseID'], edge_type='chem_disease')
for _, row in df_gd.iterrows():
    G.add_edge(row['GeneID'], row['DiseaseID'], edge_type='gene_disease')

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Graph built: 56572 nodes, 3650027 edges


In [35]:
# df_c = df_c[['ChemicalID', 'ChemicalName', 'PubChemCID']].drop_duplicates(subset=['ChemicalID']).set_index('ChemicalID')
# df_d = df_d[['DiseaseID', 'DiseaseName', 'DiseaseSemanticType']].drop_duplicates(subset=['DiseaseID']).set_index('DiseaseID')
# df_g = df_g[['GeneID', 'GeneSymbol', 'GeneName']].drop_duplicates(subset=['GeneID']).set_index('GeneID')


print(df_c.info())
print(df_g.info())
print(df_d.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 179408 entries, 0 to 179407
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   ChemicalName       179408 non-null  object
 1   ChemicalID         179408 non-null  object
 2   CasRN              56623 non-null   object
 3   Definition         6190 non-null    object
 4   ParentIDs          179407 non-null  object
 5   TreeNumbers        179408 non-null  object
 6   ParentTreeNumbers  179407 non-null  object
 7   Synonyms           116886 non-null  object
dtypes: object(8)
memory usage: 11.0+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 641489 entries, 0 to 641488
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   GeneSymbol   641488 non-null  object
 1   GeneName     639154 non-null  object
 2   GeneID       641489 non-null  int64 
 3   AltGeneIDs   48428 non-null   object
 4

In [53]:
df_g = df_g[df_g["GeneID"].isin(genes)]
df_g.info()


<class 'pandas.core.frame.DataFrame'>
Index: 28149 entries, 15907 to 641485
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   GeneSymbol   28148 non-null  object
 1   GeneName     28096 non-null  object
 2   GeneID       28149 non-null  int64 
 3   AltGeneIDs   20825 non-null  object
 4   Synonyms     26173 non-null  object
 5   BioGRIDIDs   20212 non-null  object
 6   PharmGKBIDs  21696 non-null  object
 7   UniProtIDs   20417 non-null  object
dtypes: int64(1), object(7)
memory usage: 1.9+ MB


In [51]:
paths = pd.read_csv('CTD_pathways.csv')
paths.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2567 entries, 0 to 2566
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   PathwayName  2567 non-null   object
 1   PathwayID    2567 non-null   object
dtypes: object(2)
memory usage: 40.2+ KB


In [45]:
dpath = pd.read_csv('../vocab/dispath.csv')
cpath = pd.read_csv('../vocab/chempath.csv')
gpath = pd.read_csv('../vocab/genepath.csv')

dpath['DiseaseID'] = dpath['DiseaseID'].str[5:]

cpath = cpath.drop_duplicates(subset=['ChemicalID', 'PathwayID'])  
gpath = gpath.drop_duplicates(subset=['GeneID', 'PathwayID'])
dpath = dpath.drop_duplicates(subset=['DiseaseID', 'PathwayID'])

print(dpath.head())
print(diseases)

cpath = cpath[cpath["ChemicalID"].isin(chemicals)]
gpath = gpath[gpath["GeneID"].isin(genes)]
dpath = dpath[dpath["DiseaseID"].isin(diseases)]

print(dpath.info())
print(cpath.info())
print(gpath.info())

                                  DiseaseName DiseaseID  \
0  17-Hydroxysteroid Dehydrogenase Deficiency   C537805   
1  17-Hydroxysteroid Dehydrogenase Deficiency   C537805   
2  17-Hydroxysteroid Dehydrogenase Deficiency   C537805   
3  17-Hydroxysteroid Dehydrogenase Deficiency   C537805   
4  17-Hydroxysteroid Dehydrogenase Deficiency   C537805   

                                         PathwayName            PathwayID  \
0                              Androgen biosynthesis   REACT:R-HSA-193048   
1  Fatty acid, triacylglycerol, and ketone body m...   REACT:R-HSA-535734   
2                        Fatty Acyl-CoA Biosynthesis    REACT:R-HSA-75105   
3                                 Metabolic pathways        KEGG:hsa01100   
4                                         Metabolism  REACT:R-HSA-1430728   

  InferenceGeneSymbol  
0             HSD17B3  
1             HSD17B3  
2             HSD17B3  
3             HSD17B3  
4             HSD17B3  
['D016905', 'C536789', 'D001139', 'D00

In [52]:
pathways = set(dpath['PathwayID'].unique()) | set(cpath['PathwayID'].unique())  | set(gpath['PathwayID'].unique())
len(pathways)

2363

In [46]:
# Group PathwayIDs by DiseaseID
dpaths = dpath.groupby("DiseaseID")["PathwayID"].apply(list).reset_index()
dpaths = dpaths.rename(columns={"PathwayID": "PathwayIDs"})

# Convert to list of dictionaries
result_dp = dpaths.to_dict(orient="records")

print(len(result_dp))

5051


In [47]:
# Group PathwayIDs by DiseaseID
gpaths = gpath.groupby("GeneID")["PathwayID"].apply(list).reset_index()
gpaths = gpaths.rename(columns={"PathwayID": "PathwayIDs"})

# Convert to list of dictionaries
result_gp = gpaths.to_dict(orient="records")

print(len(result_gp))

11458


In [48]:
# Group PathwayIDs by DiseaseID
cpaths = cpath.groupby("ChemicalID")["PathwayID"].apply(list).reset_index()
cpaths = cpaths.rename(columns={"PathwayID": "PathwayIDs"})

# Convert to list of dictionaries
result_cp = cpaths.to_dict(orient="records")

print(len(result_cp))

8529


In [ ]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs
import numpy as np

# Create generator once (faster)
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=1024
)

def get_ecfp_embedding(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fp = morgan_gen.GetFingerprint(mol)
    arr = np.zeros((fp.GetNumBits(),), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

count = 0
print(known.head())
print("Computing chemical embeddings...")
for chem_id in tqdm(chemicals):
    smiles = known.loc[known["meshID"] == chem_id, "smiles"].iloc[0]
    embedding = get_ecfp_embedding(smiles)
    
    G.nodes[chem_id]['embedding'] = embedding
print(count)

       meshID       cids                                             smiles
0     C070055      35823           C1=CC(=C(C=C1C2=CC(=C(C=C2Cl)Cl)Cl)Cl)Cl
1     C024713      12389                                     CCCCCCCCCCCCCC
2  D000077612     166548  CCCCCOC1=CC=C(C=C1)C2=CC=C(C=C2)C3=CC=C(C=C3)C...
3     C538809  131634859                  B(=O)OB1OB2OB(OB(O2)O1)[O-].[Na+]
4  C000626479       7565                      C1=CC=C(C=C1)OC2=CC=C(C=C2)Br
Computing chemical embeddings...


  8%|▊         | 1119/13943 [00:03<00:27, 461.62it/s][09:06:44] WARNING: not removing hydrogen atom without neighbors
[09:06:44] WARNING: not removing hydrogen atom without neighbors
 40%|████      | 5623/13943 [00:12<00:17, 475.10it/s][09:06:53] WARNING: not removing hydrogen atom without neighbors
[09:06:53] WARNING: not removing hydrogen atom without neighbors
[09:06:53] WARNING: not removing hydrogen atom without neighbors
 60%|██████    | 8400/13943 [00:18<00:18, 304.73it/s][09:06:59] WARNING: not removing hydrogen atom without neighbors
[09:06:59] WARNING: not removing hydrogen atom without neighbors
 71%|███████   | 9913/13943 [00:21<00:07, 519.27it/s][09:07:02] WARNING: not removing hydrogen atom without neighbors
[09:07:02] WARNING: not removing hydrogen atom without neighbors
[09:07:02] WARNING: not removing hydrogen atom without neighbors
[09:07:02] WARNING: not removing hydrogen atom without neighbors
 80%|███████▉  | 11106/13943 [00:23<00:04, 571.54it/s][09:07:05] WARNING:

0


In [ ]:
# 2. Disease Embeddings: Multi-hot for DiseaseSemanticType + SlimMappings + LabelEncoder on DiseaseName
all_slim_mappings = set()
disease_names = {}
diseasenodeinfo = {}
for disease_id in diseases:
    matching_rows = df_d[df_d['DiseaseID'] == disease_id]
    if not matching_rows.empty:
        row = matching_rows.iloc[0]
        # Slim mappings
        slim = str(row['SlimMappings']) if pd.notna(row['SlimMappings']) else ''
        diseasenodeinfo[disease_id] = slim.split('|') if slim else []
        all_slim_mappings.update(diseasenodeinfo[disease_id])
        # Disease name
        name = str(row['DiseaseName']) if pd.notna(row['DiseaseName']) else ''
        disease_names[disease_id] = name

# Encode semantic types
semantic_types_list = list(all_semantic_types)
semantic_encoder = LabelEncoder()
semantic_encoder.fit(semantic_types_list)
num_types = len(semantic_types_list)

# Encode slim mappings (multi-hot)
slim_mappings_list = list(all_slim_mappings)
slim_encoder = LabelEncoder()
slim_encoder.fit(slim_mappings_list)
num_slims = len(slim_mappings_list)

# Encode disease names
name_encoder = LabelEncoder()
name_encoder.fit(list(set(disease_names.values())))
name_dim = min(50, len(name_encoder.classes_))

# Total embedding dimension
total_dim = min(num_types + num_slims + name_dim, 100)

print("Computing disease embeddings...")
for disease_id in diseases:
    embedding = np.zeros(total_dim)
    matching_rows = df_d[df_d['DiseaseID'] == disease_id]
    if not matching_rows.empty:
        row = matching_rows.iloc[0]
        # Multi-hot for semantic type
        semantic_type = row['DiseaseSemanticType']
        if pd.notna(semantic_type) and num_types > 0:
            type_idx = semantic_encoder.transform([semantic_type])[0]
            if type_idx < total_dim:
                embedding[type_idx] = 1.0
        # Multi-hot for slim mappings
        slim_terms = diseasenodeinfo.get(disease_id, [])
        for slim in slim_terms:
            if slim and slim in slim_mappings_list:
                slim_idx = slim_encoder.transform([slim])[0]
                if slim_idx < total_dim - num_types:
                    embedding[num_types + slim_idx] = 1.0
        # One-hot for name
        name = disease_names.get(disease_id, '')
        if name and num_types + num_slims < total_dim:
            name_idx = name_encoder.transform([name])[0]
            if name_idx < name_dim and num_types + num_slims + name_idx < total_dim:
                embedding[num_types + num_slims + name_idx] = 1.0
    G.nodes[disease_id]['embedding'] = embedding

# ... [Gene embeddings and rest of the code unchanged] ...

In [39]:

# 2. Disease Embeddings: Multi-hot encoding of semantic types + LabelEncoder on name (e.g., 100D total)
# First, get unique semantic types for multi-hot
all_semantic_types = set()
for disease_id in diseases:
    if disease_id in df_d.index:
        semantic_type = df_d.loc[disease_id, 'DiseaseSemanticType']
        if pd.notna(semantic_type):
            all_semantic_types.add(semantic_type)
semantic_types_list = list(all_semantic_types)
semantic_encoder = LabelEncoder()
semantic_encoder.fit(semantic_types_list)
num_types = len(semantic_types_list)

# Name encoder for additional features (truncate to 50D for simplicity)
disease_names = {}
for disease_id in diseases:
    if disease_id in df_d.index:
        name = str(df_d.loc[disease_id, 'DiseaseName']) if pd.notna(df_d.loc[disease_id, 'DiseaseName']) else ''
        disease_names[disease_id] = name
name_encoder = LabelEncoder()
name_encoder.fit(list(set(disease_names.values())))
name_dim = min(50, len(name_encoder.classes_))

print("Computing disease embeddings...")
for disease_id in diseases:
    embedding = np.zeros(num_types + name_dim)
    if disease_id in df_d.index:
        # Multi-hot for semantic type
        semantic_type = df_d.loc[disease_id, 'DiseaseSemanticType']
        if pd.notna(semantic_type):
            type_idx = semantic_encoder.transform([semantic_type])[0]
            embedding[type_idx] = 1.0
        # One-hot for name (simplified to encoded index)
        name = disease_names.get(disease_id, '')
        if name:
            name_idx = name_encoder.transform([name])[0]
            if name_idx < name_dim:
                embedding[num_types + name_idx] = 1.0
    G.nodes[disease_id]['embedding'] = embedding

# 3. Gene Embeddings: One-hot on GeneSymbol + degree in graph (e.g., 100D)
# Gene degree as additional feature
gene_degrees = dict(G.degree(genes))

# Symbol encoder
all_symbols = {}
for gene_id in genes:
    if gene_id in df_g.index:
        symbol = str(df_g.loc[gene_id, 'GeneSymbol']) if pd.notna(df_g.loc[gene_id, 'GeneSymbol']) else ''
        all_symbols[gene_id] = symbol
symbol_encoder = LabelEncoder()
symbol_encoder.fit(list(set(all_symbols.values())))
symbol_dim = min(99, len(symbol_encoder.classes_))  # Leave 1 for degree

print("Computing gene embeddings...")
for gene_id in genes:
    embedding = np.zeros(symbol_dim + 1)
    # One-hot for symbol
    symbol = all_symbols.get(gene_id, '')
    if symbol:
        sym_idx = symbol_encoder.transform([symbol])[0]
        if sym_idx < symbol_dim:
            embedding[sym_idx] = 1.0
    # Degree as last feature (normalized)
    degree = gene_degrees.get(gene_id, 0)
    max_degree = max(gene_degrees.values()) if gene_degrees else 1
    embedding[-1] = degree / max_degree
    G.nodes[gene_id]['embedding'] = embedding

print("Embeddings added to all nodes.")
print("Example embeddings:")
print("Chemical example:", G.nodes[list(chemicals)[0]]['embedding'][:5] if chemicals else None)  # First 5 dims
print("Disease example:", G.nodes[list(diseases)[0]]['embedding'][:5] if diseases else None)
print("Gene example:", G.nodes[list(genes)[0]]['embedding'][:5] if genes else None)

Computing disease embeddings...
Computing gene embeddings...
Embeddings added to all nodes.
Example embeddings:
Chemical example: [0. 1. 0. 0. 1.]
Disease example: []
Gene example: [1. 0. 0. 0. 0.]
